# 金安国纪 (002636.SZ) 行情分析与技术指标

本 Notebook 演示：
1. 通过 Tushare API 获取近一年日线数据
2. 数据清洗与概览
3. K线图绘制（含均线、布林带）
4. MACD 指标计算与绘图
5. KDJ 指标计算与绘图

## 1. 环境准备与数据获取

In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# 中文字体与负号显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'PingFang SC', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ========== 配置 Tushare Token ==========
TOKEN = "ee4a8006049705f03970463dd2aeeef0ec12dd9ae51c219ceb2786a7"
ts.set_token(TOKEN)
pro = ts.pro_api()

print("Tushare 连接成功！")

In [ ]:
# ========== 获取金安国纪近一年日线数据 ==========
df = pro.daily(ts_code='002636.SZ', start_date='20250704', end_date='20260704')

# 按交易日期升序排列
df = df.sort_values('trade_date').reset_index(drop=True)

# 将 trade_date 转为 datetime 格式
df['trade_date'] = pd.to_datetime(df['trade_date'])

print(f"数据条数: {len(df)}")
print(f"日期范围: {df['trade_date'].min().strftime('%Y-%m-%d')} ~ {df['trade_date'].max().strftime('%Y-%m-%d')}")
print(f"最高价: ¥{df['high'].max():.2f} / 最低价: ¥{df['low'].min():.2f}")
print(f"最新收盘: ¥{df.iloc[-1]['close']:.2f}")
df.head(10)

In [ ]:
# ========== 数据预览 ==========
# 设定日期为索引
df.set_index('trade_date', inplace=True)

print("数据统计:")
df[['open', 'high', 'low', 'close', 'vol', 'amount']].describe()

## 2. 价格走势快速预览

In [ ]:
# ========== 收盘价走势 ==========
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df.index, df['close'], color='#e83929', linewidth=1.2, label='收盘价')
ax.fill_between(df.index, df['close'], df['close'].min(), alpha=0.1, color='#e83929')
ax.set_title('金安国纪 (002636.SZ) 收盘价走势', fontsize=16, fontweight='bold')
ax.set_ylabel('价格 (¥)', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# 标注最高最低
max_idx = df['close'].idxmax()
min_idx = df['close'].idxmin()
ax.annotate(f'最高 ¥{df["close"].max():.2f}', xy=(max_idx, df['close'].max()),
            xytext=(max_idx, df['close'].max()*1.05), fontsize=9, color='#e83929',
            arrowprops=dict(arrowstyle='->', color='#e83929'))
ax.annotate(f'最低 ¥{df["close"].min():.2f}', xy=(min_idx, df['close'].min()),
            xytext=(min_idx, df['close'].min()*0.95), fontsize=9, color='#43a047',
            arrowprops=dict(arrowstyle='->', color='#43a047'))

plt.tight_layout()
plt.show()

## 3. K线图 + 均线 + 布林带

In [ ]:
# ========== 计算均线 ==========
df['MA5']  = df['close'].rolling(5).mean().round(2)
df['MA10'] = df['close'].rolling(10).mean().round(2)
df['MA20'] = df['close'].rolling(20).mean().round(2)
df['MA60'] = df['close'].rolling(60).mean().round(2)

# ========== 计算布林带 (20, 2) ==========
df['BOLL_MID'] = df['MA20']
df['BOLL_STD'] = df['close'].rolling(20).std().round(4)
df['BOLL_UP']  = (df['BOLL_MID'] + 2 * df['BOLL_STD']).round(2)
df['BOLL_DN']  = (df['BOLL_MID'] - 2 * df['BOLL_STD']).round(2)

print("均线与布林带计算完成！\n")
df[['close', 'MA5', 'MA10', 'MA20', 'MA60', 'BOLL_UP', 'BOLL_MID', 'BOLL_DN']].tail(10)

In [ ]:
# ========== 绘制 K 线图 ==========
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(16, 7))

def plot_candlestick(ax, df_data, width=0.6):
    """手绘 K 线蜡烛图"""
    up_color = '#e83929'   # 涨-红色 (中国惯例)
    down_color = '#43a047' # 跌-绿色
    
    for i, (idx, row) in enumerate(df_data.iterrows()):
        open_, high, low, close_ = row['open'], row['high'], row['low'], row['close']
        color = up_color if close_ >= open_ else down_color
        
        # 影线
        ax.plot([i, i], [low, high], color=color, linewidth=0.8)
        # 实体
        body_h = abs(close_ - open_)
        body_bottom = min(open_, close_)
        if body_h > 0:
            rect = Rectangle((i - width/2, body_bottom), width, body_h,
                            facecolor=color, edgecolor=color, linewidth=0.5)
            ax.add_patch(rect)
        else:
            ax.plot([i - width/2, i + width/2], [close_, close_], color=color, linewidth=1.5)

plot_candlestick(ax, df, width=0.7)

# 均线
ax.plot(range(len(df)), df['MA5'],  linewidth=1, color='#f5a623', label='MA5')
ax.plot(range(len(df)), df['MA10'], linewidth=1, color='#4a90d9', label='MA10')
ax.plot(range(len(df)), df['MA20'], linewidth=1.2, color='#9b59b6', label='MA20')
ax.plot(range(len(df)), df['MA60'], linewidth=1, color='#2ecc71', label='MA60')

# 布林带
ax.plot(range(len(df)), df['BOLL_UP'], linewidth=0.8, color='#d4a574',
        linestyle='--', label='BOLL上轨', alpha=0.7)
ax.plot(range(len(df)), df['BOLL_DN'], linewidth=0.8, color='#d4a574',
        linestyle='--', label='BOLL下轨', alpha=0.7)
ax.fill_between(range(len(df)), df['BOLL_UP'], df['BOLL_DN'],
                alpha=0.05, color='#d4a574')

# 格式化 X 轴
tick_step = max(1, len(df) // 10)
tick_idx = range(0, len(df), tick_step)
tick_labels = [df.index[i].strftime('%Y-%m') for i in tick_idx]
ax.set_xticks(tick_idx)
ax.set_xticklabels(tick_labels, fontsize=9, rotation=30)

ax.set_title('金安国纪 (002636.SZ) K线图 · MA均线 · BOLL布林带', fontsize=16, fontweight='bold')
ax.set_ylabel('价格 (¥)', fontsize=12)
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.grid(True, alpha=0.2)
ax.set_xlim(-1, len(df))

plt.tight_layout()
plt.show()

## 4. MACD 指标

In [ ]:
# ========== 计算 MACD (12, 26, 9) ==========
def ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

df['EMA12'] = ema(df['close'], 12)
df['EMA26'] = ema(df['close'], 26)
df['DIF']   = (df['EMA12'] - df['EMA26']).round(4)
df['DEA']   = ema(df['DIF'], 9).round(4)
df['MACD']  = (2 * (df['DIF'] - df['DEA'])).round(4)

# 最新信号
last = df.iloc[-1]
signal = '多头 (看涨)' if last['MACD'] > 0 else '空头 (看跌)'
print(f"最新 MACD 信号: {signal}")
print(f"  DIF: {last['DIF']:.2f}")
print(f"  DEA: {last['DEA']:.2f}")
print(f"  MACD柱: {last['MACD']:.2f}")

df[['close', 'DIF', 'DEA', 'MACD']].tail(10)

In [ ]:
# ========== 绘制 MACD 图 ==========
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

# 上: 收盘价
ax1.plot(df.index, df['close'], color='#333', linewidth=1.2, label='收盘价')
ax1.set_ylabel('价格 (¥)', fontsize=12)
ax1.set_title('金安国纪 (002636.SZ) 收盘价 & MACD 指标', fontsize=16, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# 下: MACD
ax2.plot(df.index, df['DIF'],  linewidth=1, color='#e83929', label='DIF')
ax2.plot(df.index, df['DEA'],  linewidth=1, color='#4a90d9', label='DEA')
# MACD 柱
colors = ['#e83929' if v >= 0 else '#43a047' for v in df['MACD']]
ax2.bar(df.index, df['MACD'], width=0.8, color=colors, alpha=0.8, label='MACD柱')

ax2.axhline(y=0, color='#666', linewidth=0.5, linestyle='-')
ax2.set_ylabel('MACD', fontsize=12)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. KDJ 指标

In [ ]:
# ========== 计算 KDJ (9, 3, 3) ==========
def calc_kdj(df, n=9, m1=3, m2=3):
    """
    计算 KDJ 指标
    n: RSV 周期 (默认9)
    m1: K 平滑系数 (默认3)
    m2: D 平滑系数 (默认3)
    """
    low_n  = df['low'].rolling(n).min()
    high_n = df['high'].rolling(n).max()
    
    # RSV
    rsv = ((df['close'] - low_n) / (high_n - low_n) * 100).fillna(50)
    
    # K, D, J
    k = rsv.ewm(alpha=1/m1, adjust=False).mean()
    d = k.ewm(alpha=1/m2, adjust=False).mean()
    j = 3 * k - 2 * d
    
    return k.round(2), d.round(2), j.round(2)

df['K'], df['D'], df['J'] = calc_kdj(df)

# 最新信号
last_k, last_d, last_j = df.iloc[-1][['K','D','J']]
if last_k > 80:
    kdj_signal = '超买区域 (注意回调风险)'
elif last_k < 20:
    kdj_signal = '超卖区域 (可能反弹)'
else:
    kdj_signal = '中性区域'

print(f"最新 KDJ 信号: {kdj_signal}")
print(f"  K: {last_k:.1f}")
print(f"  D: {last_d:.1f}")
print(f"  J: {last_j:.1f}")

df[['close', 'K', 'D', 'J']].tail(10)

In [ ]:
# ========== 绘制 KDJ 图 ==========
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(df.index, df['K'], linewidth=1, color='#f5a623', label='K')
ax.plot(df.index, df['D'], linewidth=1, color='#4a90d9', label='D')
ax.plot(df.index, df['J'], linewidth=1, color='#9b59b6', label='J')

# 超买/超卖区域
ax.axhline(y=80, color='#e83929', linewidth=0.5, linestyle='--', alpha=0.5)
ax.axhline(y=20, color='#43a047', linewidth=0.5, linestyle='--', alpha=0.5)
ax.fill_between(df.index, 80, 100, alpha=0.05, color='#e83929')
ax.fill_between(df.index, 0, 20, alpha=0.05, color='#43a047')
ax.text(df.index[0], 82, '超买区 80', fontsize=8, color='#e83929', alpha=0.7)
ax.text(df.index[0], 18, '超卖区 20', fontsize=8, color='#43a047', alpha=0.7)

ax.set_ylim(0, 100)
ax.set_title('金安国纪 (002636.SZ) KDJ 指标 (9,3,3)', fontsize=16, fontweight='bold')
ax.set_ylabel('KDJ', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 成交量分析

In [ ]:
# ========== 成交量与价格叠加 ==========
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

# 收盘价
ax1.plot(df.index, df['close'], color='#e83929', linewidth=1.2)
ax1.set_ylabel('收盘价 (¥)', fontsize=12)
ax1.set_title('金安国纪 (002636.SZ) 价格 & 成交量', fontsize=16, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 成交量柱状图
vol_colors = ['#e83929' if df.iloc[i]['close'] >= df.iloc[i-1]['close'] if i > 0 else df.iloc[i]['close'] >= df.iloc[i]['open']
              else '#43a047' for i in range(len(df))]
ax2.bar(df.index, df['vol'] / 10000, width=0.8, color=vol_colors, alpha=0.8)
ax2.set_ylabel('成交量 (万手)', fontsize=12)
ax2.grid(True, alpha=0.3)

# 标记最大成交量日
max_vol_idx = df['vol'].idxmax()
max_vol = df.loc[max_vol_idx, 'vol'] / 10000
ax2.annotate(f'最大量 {max_vol:.0f}万手', xy=(max_vol_idx, max_vol),
            xytext=(max_vol_idx, max_vol*0.8), fontsize=9, color='#e83929',
            arrowprops=dict(arrowstyle='->', color='#e83929'))

plt.tight_layout()
plt.show()

## 7. 全景综合图

将 K线+均线+布林带、成交量、MACD、KDJ 四图合一展示

In [ ]:
# ========== 全景综合图 ==========
fig, axes = plt.subplots(4, 1, figsize=(16, 12),
                         gridspec_kw={'height_ratios': [3, 1.2, 1.2, 1.2]}, sharex=True)

ax_kline, ax_vol, ax_macd, ax_kdj = axes

# ---- 图1: K线 + 均线 + 布林带 ----
plot_candlestick(ax_kline, df, width=0.7)
ax_kline.plot(range(len(df)), df['MA5'],  linewidth=1, color='#f5a623', label='MA5')
ax_kline.plot(range(len(df)), df['MA10'], linewidth=1, color='#4a90d9', label='MA10')
ax_kline.plot(range(len(df)), df['MA20'], linewidth=1.2, color='#9b59b6', label='MA20')
ax_kline.plot(range(len(df)), df['MA60'], linewidth=1, color='#2ecc71', label='MA60')
ax_kline.plot(range(len(df)), df['BOLL_UP'], linewidth=0.8, color='#d4a574', linestyle='--', alpha=0.6)
ax_kline.plot(range(len(df)), df['BOLL_DN'], linewidth=0.8, color='#d4a574', linestyle='--', alpha=0.6)
ax_kline.fill_between(range(len(df)), df['BOLL_UP'], df['BOLL_DN'], alpha=0.04, color='#d4a574')

tick_step = max(1, len(df) // 10)
tick_idx = range(0, len(df), tick_step)
ax_kline.set_xticks(tick_idx)
ax_kline.set_ylabel('价格 (¥)', fontsize=10)
ax_kline.set_title('金安国纪 (002636.SZ) 全景分析', fontsize=16, fontweight='bold')
ax_kline.legend(loc='upper left', fontsize=7, ncol=3)
ax_kline.grid(True, alpha=0.2)
ax_kline.set_xlim(-1, len(df))

# ---- 图2: 成交量 ----
ax_vol.bar(range(len(df)), df['vol']/10000, width=0.8, color=vol_colors, alpha=0.8)
ax_vol.set_ylabel('量(万手)', fontsize=10)
ax_vol.grid(True, alpha=0.3)

# ---- 图3: MACD ----
ax_macd.plot(range(len(df)), df['DIF'],  linewidth=1, color='#e83929')
ax_macd.plot(range(len(df)), df['DEA'],  linewidth=1, color='#4a90d9')
macd_colors = ['#e83929' if v >= 0 else '#43a047' for v in df['MACD']]
ax_macd.bar(range(len(df)), df['MACD'], width=0.8, color=macd_colors, alpha=0.8)
ax_macd.axhline(y=0, color='#666', linewidth=0.5)
ax_macd.set_ylabel('MACD', fontsize=10)
ax_macd.grid(True, alpha=0.3)

# ---- 图4: KDJ ----
ax_kdj.plot(range(len(df)), df['K'], linewidth=1, color='#f5a623')
ax_kdj.plot(range(len(df)), df['D'], linewidth=1, color='#4a90d9')
ax_kdj.plot(range(len(df)), df['J'], linewidth=1, color='#9b59b6')
ax_kdj.axhline(y=80, color='#e83929', linewidth=0.5, linestyle='--', alpha=0.5)
ax_kdj.axhline(y=20, color='#43a047', linewidth=0.5, linestyle='--', alpha=0.5)
ax_kdj.set_ylim(0, 100)
ax_kdj.set_ylabel('KDJ', fontsize=10)
ax_kdj.grid(True, alpha=0.3)

# X轴标签
tick_labels = [df.index[i].strftime('%Y-%m') for i in tick_idx]
ax_kdj.set_xticks(tick_idx)
ax_kdj.set_xticklabels(tick_labels, fontsize=9, rotation=30)

plt.tight_layout()
plt.show()

## 8. 总结

本 Notebook 完成了以下工作：

- ✅ 通过 Tushare API 获取金安国纪 (002636.SZ) 近一年日线数据
- ✅ K线图手绘，含 MA5/10/20/60 均线
- ✅ BOLL 布林带（中轨=MA20，带宽=2倍标准差）
- ✅ MACD (12, 26, 9) 指标计算与双图展示
- ✅ KDJ (9, 3, 3) 指标计算与超买超卖标注
- ✅ 成交量柱状图红涨绿跌
- ✅ 四图合一全景分析

> 注意：Token 已硬编码在 Notebook 中，如需分享请先移除。